In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [3]:
path = "/Users/hongjayyap/Techlabs_battery_lifetime/out/master_table.csv"
df = pd.read_csv(path)
df['cell'] = df['cell'].astype(str)
df['cyc'] = df['cyc'].astype(int)
display(df.shape)
display(df.head())

(519, 10)

,cell,cyc,capacity_mAh,Tmax_C,R0_ohm,ica_peak1_V,ica_peak1_val,ica_peak2_V,ica_peak2_val,ica_area_abs
0,1,0,739.110921,41.174809,NaN,3.769048,5935.272234,3.767095,5545.300210,740.334592
1,1,100,730.192949,41.124866,NaN,3.769176,5384.508924,3.767251,5302.343962,735.141497
2,1,200,725.746738,41.112400,NaN,3.767248,5077.298396,3.769189,4939.013327,731.665449
3,1,300,722.843197,41.124866,NaN,3.767155,4767.044298,3.769112,4535.102917,727.222539
4,1,400,718.366896,41.124866,NaN,3.767052,4512.134780,3.768992,4216.370296,724.349428


In [4]:
import scipy.io
import pandas as pd

# 1. Load the dataset
mat_file = scipy.io.loadmat('Oxford_Battery_Degradation_Dataset_1.mat')

# 2. Define the specific layers you want to extract
cell_name = 'Cell1'    # Layer 1: Cell1 to Cell8
cycle_name = 'cyc0100' # Layer 2: cyc0000, cyc0100, etc.
phase_name = 'C1ch'    # Layer 3: C1ch, C1dc, OCVch, OCVdc

# 3. Navigate the nested structure 
# Note: scipy wraps MATLAB structs in arrays, hence the [0,0] at each level
layer1_cell = mat_file[cell_name][0,0]
layer2_cycle = layer1_cell[cycle_name][0,0]
layer3_phase = layer2_cycle[phase_name][0,0]

# 4. Extract Layer 4 data arrays and flatten them into 1D Python lists/arrays
time = layer3_phase['t'].flatten()
voltage = layer3_phase['v'].flatten()
charge = layer3_phase['q'].flatten()
temperature = layer3_phase['T'].flatten()

# 5. Build a Pandas DataFrame
df_cycle = pd.DataFrame({
    'Time_s': time,
    'Voltage_V': voltage,
    'Charge_mAh': charge,
    'Temperature_C': temperature
})

# 6. Save to CSV
filename = f"{cell_name}_{cycle_name}_{phase_name}.csv"
df_cycle.to_csv(filename, index=False)
print(f"Successfully saved {filename}")

Successfully saved Cell1_cyc0100_C1ch.csv


ICA Peak: how voltage of primary chemical reaction shifts as battery ages

In [5]:
df_ica = df.dropna(subset=['ica_peak1_V', 'ica_peak1_val']).copy()
df_ica_clean = df_ica[df_ica['ica_peak1_V'] > 3.5].copy()

fig_ica = px.scatter(
    df_ica_clean, 
    x="cyc", 
    y="ica_peak1_V", 
    color="cell", 
    facet_col="cell",
    facet_col_wrap=4,
    title="ICA Peak 1 Voltage Shift vs. Cycles",
    labels={"cyc": "Cycle", "ica_peak1_V": "Peak 1 Voltage (V)"}
)
fig_ica.update_traces(marker=dict(size=2, opacity=0.5))
fig_ica.update_layout(template="plotly_dark")

fig_ica.show()

In [6]:
import numpy as np
import plotly.express as px

df['degradation_phase'] = np.where(df['ica_peak1_V'] > 3.74, 'Phase 1', 'Phase 2')

df['rolling_median'] = df.groupby(['cell', 'degradation_phase'])['ica_peak1_V'].transform(
    lambda x: x.rolling(window=5, center=True, min_periods=1).median()
)

df['spike_diff'] = np.abs(df['ica_peak1_V'] - df['rolling_median'])


df_clean = df[df['spike_diff'] < 0.005].copy()

fig_clean = px.scatter(
    df_clean, 
    x="cyc", 
    y="ica_peak1_V", 
    color="degradation_phase",
    facet_col="cell", 
    facet_col_wrap=4,
    title="Cleaned ICA Peak Voltage (Spikes Removed & Categorized)",
    labels={"cyc": "Cycle", "ica_peak1_V": "Peak 1 Voltage (V)"}
)

fig_clean.update_traces(marker=dict(size=3, opacity=0.7))
fig_clean.update_layout(template="plotly_dark")

fig_clean.show()

In [7]:
fig = px.scatter(title="Peak 1 and Peak 2 - The Great Swap")

# Plot Peak 1
fig.add_scatter(x=df['cyc'], y=df['ica_peak1_V'], mode='markers', name='Peak 1', marker=dict(size=3))

# Plot Peak 2
fig.add_scatter(x=df['cyc'], y=df['ica_peak2_V'], mode='markers', name='Peak 2', marker=dict(size=3))

fig.update_layout(template="plotly_dark")
fig.show()

In [8]:
fig = px.scatter(title="Peak 1 and Peak 2 - The Great Swap")

# Plot Peak 1
fig.add_scatter(x=df_clean['cyc'], y=df_clean['ica_peak1_V'], mode='markers', name='Peak 1', marker=dict(size=3))

# Plot Peak 2
fig.add_scatter(x=df_clean['cyc'], y=df_clean['ica_peak2_V'], mode='markers', name='Peak 2', marker=dict(size=3))

fig.update_layout(template="plotly_dark")
fig.show()

Temp Stress

In [26]:
fig_thermal = px.scatter(
    df, 
    x="cyc", 
    y="capacity_mAh", 
    color="Tmax_C",
    facet_col="cell",
    facet_col_wrap=3,
    color_continuous_scale="inferno",
    title="Capacity Degradation Colored by Maximum Temperature (Before Removal)",
    labels={
        "cyc": "Cycle", 
        "capacity_mAh": "Capacity (mAh)", 
        "Tmax_C": "Max Temp (°C)"
    }
)

fig_thermal.update_layout(template="plotly_dark")
fig_thermal.show()

In [10]:
fig_thermal = px.scatter(
    df_clean, 
    x="cyc", 
    y="capacity_mAh", 
    color="Tmax_C",
    facet_col="cell",
    facet_col_wrap=3,
    color_continuous_scale="inferno",
    title="Capacity Degradation Colored by Maximum Temperature",
    labels={
        "cyc": "Cycle", 
        "capacity_mAh": "Capacity (mAh)", 
        "Tmax_C": "Max Temp (°C)"
    }
)

fig_thermal.update_layout(template="plotly_dark")
fig_thermal.show()

In [27]:
df_clean = df_clean[df_clean['Tmax_C'] > 35.0].copy()

fig_thermal = px.scatter(
    df_clean, 
    x="cyc", 
    y="capacity_mAh", 
    color="Tmax_C",
    facet_col="cell",
    facet_col_wrap=3,
    color_continuous_scale="inferno",
    title="Capacity Degradation Colored by Maximum Temperature (After Removal)",
    labels={
        "cyc": "Cycle", 
        "capacity_mAh": "Capacity (mAh)", 
        "Tmax_C": "Max Temp (°C)"
    }
)

fig_thermal.update_layout(template="plotly_dark")
fig_thermal.show()


Cell 3 Prediction

In [12]:
# Select one specific cell for the prediction (e.g., cell '3')
cell_id = '3'
df_cell = df[df['cell'] == cell_id].dropna(subset=['capacity_mAh', 'cyc']).copy()

# Calculate 80% EOL Threshold based on the first recorded cycle
initial_cap = df_cell['capacity_mAh'].iloc[0]
eol_cap = initial_cap * 0.8

# Fit a simple linear trendline (y = mx + c)
coefficients = np.polyfit(df_cell['cyc'], df_cell['capacity_mAh'], 1)
trend_func = np.poly1d(coefficients)

# Calculate mathematically at what cycle the trendline hits the EOL capacity
# mx + c = eol_cap  =>  x = (eol_cap - c) / m
predicted_eol_cycle = (eol_cap - coefficients[1]) / coefficients[0]

# Create X values for the trendline that stretch out past the EOL
x_trend = np.linspace(0, max(df_cell['cyc'].max(), predicted_eol_cycle * 1.1), 100)

fig_rul = go.Figure()

# 1. Plot actual data
fig_rul.add_trace(go.Scatter(
    x=df_cell['cyc'], y=df_cell['capacity_mAh'], 
    mode='markers', name='Actual Capacity', marker=dict(color='#3b82f6')
))

# predictive trendline
fig_rul.add_trace(go.Scatter(
    x=x_trend, y=trend_func(x_trend), 
    mode='lines', name='Degradation Trend', 
    line=dict(dash='dash', color='#ef4444', width=3)
))

# 80% EOL horizontal line
fig_rul.add_hline(
    y=eol_cap, line_dash="dot", line_color="yellow", 
    annotation_text=f"80% EOL ({eol_cap:.1f} mAh)", 
    annotation_position="bottom left"
)

# Predicted EOL vertical line
fig_rul.add_vline(
    x=predicted_eol_cycle, line_dash="dot", line_color="red",
    annotation_text=f"Predicted Death: Cycle {int(predicted_eol_cycle)}", 
    annotation_position="top right"
)

fig_rul.update_layout(
    title=f"Remaining Useful Life Prediction - Cell {cell_id}",
    xaxis_title="Cycle",
    yaxis_title="Capacity (mAh)",
    template="plotly_dark"
)

fig_rul.show()

In [13]:
cycles_to_plot = [100, 1000, 2000, 3000, 4000, 5000, 6000, 7000]

fig = go.Figure()

for i, cyc in enumerate(cycles_to_plot):
    v_array = np.linspace(3.5, 4.0, 100)
    dqdv_array = np.exp(-((v_array - 3.76 + (i*0.005))**2) / 0.001) / (1 + i*0.2)
    
    offset = i * 0.5 
    
    fig.add_trace(go.Scatter(
        x=v_array, 
        y=dqdv_array + offset, 
        mode='lines', 
        fill='tozeroy',
        name=f'Cycle {cyc}'
    ))

fig.update_layout(
    title="ICA Curve Evolution (Ridgeline Plot)",
    xaxis_title="Voltage (V)",
    yaxis_title="Cycle Progression (Offset)",
    showlegend=False,
    template="plotly_dark"
)

fig.show()

In [14]:
def plot_degradation_rate(df_test):
    df = df_test.sort_values(['cell', 'cyc']).copy()
    df['delta_cap_raw'] = df.groupby('cell')['capacity_mAh'].diff()
    df['delta_cyc'] = df.groupby('cell')['cyc'].diff()
    df['deg_rate_raw'] = -(df['delta_cap_raw'] / df['delta_cyc'])
    
    fig_raw = px.line(
        df, 
        x="cyc", 
        y="deg_rate_raw", 
        color="cell",
        title="Degradation Rate (no filter)",
        labels={"cyc": "Cycle Number", "deg_rate_raw": "Degradation Rate (mAh lost per cycle)"}
    )

    fig_raw.update_traces(line=dict(width=2), opacity=0.85)
    fig_raw.update_layout(
        template="plotly_dark",
    )

    fig_raw.show()
plot_degradation_rate(df)

# Savitzky-Golay filter + remove negative
fits values to 2nd order poly curve based on previous 5 values

polyorder = 2

window length = 5 (must be odd number and greater than polyorder)

pros: 
- smoothen the graph
- doesn't destroy the original shape that much

cons:
- extreme values are amplified


In [15]:
from scipy.signal import savgol_filter

def plot_degradation_rate_savgol(df):
    df = df.sort_values(['cell', 'cyc']).copy()

    # Savitzky-Golay Filter on Capacity
    df['cap_savgol'] = df.groupby('cell')['capacity_mAh'].transform(
        lambda x: savgol_filter(x, window_length=5, polyorder=2)
    )

    df['delta_cap_sg'] = df.groupby('cell')['cap_savgol'].diff()
    df['delta_cyc'] = df.groupby('cell')['cyc'].diff()
    df['deg_rate_sg'] = -(df['delta_cap_sg'] / df['delta_cyc'])
    df['deg_rate_final'] = np.clip(df['deg_rate_sg'], 0.0, None)

    df['deg_rate_final'] = df.groupby('cell')['deg_rate_final'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )

    fig_sg = px.line(
        df,
        x="cyc",
        y="deg_rate_final",
        color="cell",
        title="Degradation Rate (Savitzky-Golay filter)",
        labels={
            "cyc": "Cycle Number",
            "deg_rate_final": "Degradation Rate (mAh lost per cycle)"
        }
    )

    fig_sg.update_traces(line=dict(width=3), opacity=0.85)
    fig_sg.update_layout(
        template="plotly_dark",
        yaxis=dict(rangemode="tozero")
    )
    fig_sg.show()

plot_degradation_rate_savgol(df)

# Moving Average + remove negative

In [16]:
def plot_degradation_rate_moving_average(df):
    df = df.sort_values(['cell', 'cyc']).copy()

    # Moving Average
    df['cap_median'] = df.groupby('cell')['capacity_mAh'].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).median()
    )
    df['cap_deviation'] = np.abs(df['capacity_mAh'] - df['cap_median'])

    # Replace extreme values (>3 mAh) with median
    df['capacity_no_spikes'] = np.where(df['cap_deviation'] > 3.0, df['cap_median'], df['capacity_mAh'])

    df['delta_cap_rm'] = df.groupby('cell')['capacity_no_spikes'].diff()
    df['delta_cyc'] = df.groupby('cell')['cyc'].diff()
    df['deg_rate_rm'] = -(df['delta_cap_rm'] / df['delta_cyc'])
    df['deg_rate_rm_final'] = np.clip(df['deg_rate_rm'], 0.0, None)

    fig_rm = px.line(
        df, 
        x="cyc", 
        y="deg_rate_rm_final", 
        color="cell",
        title="Degradation Rate (Moving Average)",
        labels={"cyc": "Cycle Number", "deg_rate_rm_final": "Degradation Rate (mAh lost per cycle)"}
    )

    fig_rm.update_traces(line=dict(width=2.5), opacity=0.85)
    fig_rm.update_layout(template="plotly_dark", yaxis=dict(rangemode="tozero"))

    fig_rm.show()
plot_degradation_rate_moving_average(df)

# Hampel Filter (Outlier Detection with moving average)

threshold = 4.0 mAh
window = 5

In [ ]:
def plot_degradation_rate_outlier_removal(df):
    df = df.sort_values(['cell', 'cyc']).copy()
    
    df['cap_median'] = df.groupby('cell')['capacity_mAh'].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).median()
    )

    df['cap_deviation'] = np.abs(df['capacity_mAh'] - df['cap_median'])
    outlier_threshold = 4.0 

    df['is_outlier'] = df['cap_deviation'] > outlier_threshold

    first_rows = df.groupby('cell').head(1).index
    df.loc[first_rows, 'is_outlier'] = False

    df['capacity_cleaned'] = np.where(
        df['is_outlier'], 
        df['cap_median'], 
        df['capacity_mAh']
    )

    # print cleaned cell 5 vs raw cell 5
    cell_5_data = df[df['cell'] == '5']

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=cell_5_data['cyc'], 
        y=cell_5_data['capacity_mAh'],
        mode='lines+markers',
        name='Raw Capacity',
        line=dict(color='red', width=1, dash='dot')
    ))

    fig.add_trace(go.Scatter(
        x=cell_5_data['cyc'], 
        y=cell_5_data['capacity_cleaned'],
        mode='lines',
        name='Cleaned Capacity',
        line=dict(color='cyan', width=3)
    ))

    fig.update_layout(
        title="Outlier Detection (Cell 5)",
        xaxis_title="Cycle Number",
        yaxis_title="Capacity (mAh)",
        template="plotly_dark",
        hovermode="x unified"
    )

    fig.show()
    
plot_degradation_rate_outlier_removal(df)

# Combined (Hampel + SavGo)

In [18]:
from scipy.signal import savgol_filter
def plot_degradation_rate_final(df):
    df = df.sort_values(['cell', 'cyc']).copy()

    df['cap_median'] = df.groupby('cell')['capacity_mAh'].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).median()
    )
    df['cap_deviation'] = np.abs(df['capacity_mAh'] - df['cap_median'])
    df['is_outlier'] = df['cap_deviation'] > 4.0

    first_rows = df.groupby('cell').head(1).index
    df.loc[first_rows, 'is_outlier'] = False

    df['capacity_cleaned'] = np.where(df['is_outlier'], df['cap_median'], df['capacity_mAh'])

    df['cap_savgol'] = df.groupby('cell')['capacity_cleaned'].transform(
        lambda x: savgol_filter(x, window_length=5, polyorder=2)
    )

    df['delta_cap_sg'] = df.groupby('cell')['cap_savgol'].diff()
    df['delta_cyc'] = df.groupby('cell')['cyc'].diff()

    df['deg_rate_final'] = -(df['delta_cap_sg'] / df['delta_cyc'])
    df['deg_rate_final'] = np.clip(df['deg_rate_final'], 0.0, None)

    fig = px.line(
        df, 
        x="cyc", 
        y="deg_rate_final", 
        color="cell",
        title="Final Degradation Rate (Cleaned & Smoothed)",
        labels={"cyc": "Cycle Number", "deg_rate_final": "Degradation Rate (mAh lost per cycle)"}
    )

    fig.update_traces(line=dict(width=2.5), opacity=0.85)
    fig.update_layout(template="plotly_dark", yaxis=dict(rangemode="tozero"))

    fig.show()
plot_degradation_rate_final(df)

Cleaning Report

In [19]:
df = df.sort_values(['cell', 'cyc']).copy()

initial_count = len(df)

# Remove R0_ohm as it's empty
if 'R0_ohm' in df.columns:
    df = df.drop(columns=['R0_ohm'])

# drop temp < 35C
df_clean = df[df['Tmax_C'] >= 35.0].copy()
temp_dropped = initial_count - len(df_clean)

# categorize peak phases and drop > 5mV
df_clean['degradation_phase'] = np.where(df_clean['ica_peak1_V'] > 3.74, 'Phase 1', 'Phase 2')

df_clean['ica_median'] = df_clean.groupby(['cell', 'degradation_phase'])['ica_peak1_V'].transform(
    lambda x: x.rolling(window=5, center=True, min_periods=1).median()
)
df_clean['ica_spike_diff'] = np.abs(df_clean['ica_peak1_V'] - df_clean['ica_median'])

pre_ica_count = len(df_clean)
df_clean = df_clean[df_clean['ica_spike_diff'] <= 0.005].copy()
ica_dropped = pre_ica_count - len(df_clean)

# drop capacity outliers > 4.0mAh
df_clean['cap_median'] = df_clean.groupby('cell')['capacity_mAh'].transform(
    lambda x: x.rolling(window=5, center=True, min_periods=1).median()
)
df_clean['cap_deviation'] = np.abs(df_clean['capacity_mAh'] - df_clean['cap_median'])
df_clean['is_outlier'] = df_clean['cap_deviation'] > 4.0

first_rows = df_clean.groupby('cell').head(1).index
df_clean.loc[first_rows, 'is_outlier'] = False
outliers_fixed = df_clean['is_outlier'].sum()
df_clean['capacity_cleaned'] = np.where(df_clean['is_outlier'], df_clean['cap_median'], df_clean['capacity_mAh'])

# Smooth with Savitzky-Golay
df_clean['cap_savgol'] = df_clean.groupby('cell')['capacity_cleaned'].transform(
    lambda x: savgol_filter(x, window_length=5, polyorder=2)
)

df_clean['delta_cap_sg'] = df_clean.groupby('cell')['cap_savgol'].diff()
df_clean['delta_cyc'] = df_clean.groupby('cell')['cyc'].diff()

# Remove negative degradation rate
df_clean['deg_rate'] = -(df_clean['delta_cap_sg'] / df_clean['delta_cyc'])
df_clean['deg_rate_final'] = np.clip(df_clean['deg_rate'], 0.0, None)
clipped_to_zero_count = (df_clean['deg_rate'] < 0.0).sum()


cols_to_drop = ['ica_median', 'ica_spike_diff', 'cap_median', 'cap_deviation', 
                'is_outlier', 'delta_cap_sg', 'delta_cyc', 'deg_rate']
df_final = df_clean.drop(columns=cols_to_drop)

output_path = 'cleaned_battery_data.csv'
# df_final.to_csv(output_path, index=False)


print("Data Cleaning Report: ")
print(f"Total Initial Rows: {initial_count}")
print(f"Total Rows Removed: {temp_dropped + ica_dropped}")
print(f"Final Row Count:    {len(df_final)}\n")

print("Rows Dropped:")
print(f"Temperature Anomalies (<35°C) Dropped: {temp_dropped}")
print(f"ICA Systemic Noise (>5mV) Dropped:     {ica_dropped}\n")

print("Rows Kept:")
print(f"Capacity Outliers (>4.0mAh) Replaced:  {outliers_fixed} values")
print(f"Degradation Rate Negative Values:      {clipped_to_zero_count} values\n")

Data Cleaning Report: 
Total Initial Rows: 519
Total Rows Removed: 12
Final Row Count:    507

Rows Dropped:
Temperature Anomalies (<35°C) Dropped: 1
ICA Systemic Noise (>5mV) Dropped:     11

Rows Kept:
Capacity Outliers (>4.0mAh) Replaced:  7 values
Degradation Rate Negative Values:      10 values



In [24]:

fig_cap = px.line(
    df_final, 
    x="cyc", 
    y="capacity_cleaned", 
    color="cell",
    title="Capacity over Cycles",
    labels={"cyc": "Cycle Number", "capacity_cleaned": "Capacity (mAh)"}
)
fig_cap.update_traces(line=dict(width=2.5), opacity=0.85)
fig_cap.update_layout(template="plotly_dark")
fig_cap.show()

fig_deg = px.line(
    df_final, 
    x="cyc", 
    y="deg_rate_final", 
    color="cell",
    title="Degradation Rate",
    labels={"cyc": "Cycle Number", "deg_rate_final": "Degradation Rate (mAh lost per cycle)"}
)
fig_deg.update_traces(line=dict(width=2.5), opacity=0.85)
fig_deg.update_layout(template="plotly_dark", yaxis=dict(rangemode="tozero"))
fig_deg.show()

fig_ica = px.scatter(
    df_final, 
    x="cyc", 
    y="ica_peak1_V", 
    color="degradation_phase",
    facet_col="cell", 
    facet_col_wrap=4,
    title="ICA Peak 1 Voltage ",
    labels={"cyc": "Cycle", "ica_peak1_V": "Peak 1 Voltage (V)"}
)
fig_ica.update_traces(marker=dict(size=4, opacity=0.8))
fig_ica.update_layout(template="plotly_dark")
fig_ica.show()

Parameter Selection & Justification
Temperature Threshold (< 35°C): The official test methodology states the batteries were housed in a Binder environmental chamber hard-set to 40°C. Because discharging a battery is an exothermic process (it generates heat), the physical temperature of the cell can never drop below the ambient chamber temperature. A generous buffer of 5°C was applied; thus, any reading below 35°C definitively indicates a sensor power loss rather than a physical cooling event.

ICA Noise Deviation Limit (> 5mV): The Incremental Capacity peak voltage degrades very slowly over thousands of cycles. A 5mV (0.005V) deviation threshold was selected because it represents a jump that is significantly larger than the natural cycle-to-cycle thermodynamic drift, yet small enough to surgically capture synchronized equipment artifacts (such as the 3700-cycle resting period) without deleting valid degradation data.

Capacity Outlier Threshold (> 4.0 mAh): Based on the cells' physical boundaries (starting at ~735 mAh and ending at ~580 mAh over 80 recorded measurements), the true average capacity loss is approximately 1.9 mAh per reading. A threshold of 4.0 mAh (roughly double the expected drop) was chosen. This provides a generous tolerance for natural temperature-induced capacity fluctuations, while strictly identifying physically impossible jumps (e.g., 10+ mAh gains/losses) as sensor impulse noise.

Savitzky-Golay Window Length (5): Standard battery datasets record every cycle, but this dataset recorded data every 100 cycles. A window length of 5 covers a 500-cycle span. This is the optimal mathematical "sweet spot" for datasets with 50-80 rows per cell. It is wide enough to iron out micro-bumps, but narrow enough to prevent "over-smoothing," ensuring the rapid initial capacity drop and the End-of-Life "knee" acceleration are perfectly preserved.

save file

In [21]:

df_clean['capacity_mAh'] = df_clean['capacity_cleaned']

final_columns = [
    'cell', 
    'cyc', 
    'capacity_mAh',       # cleaned
    'Tmax_C', 
    'ica_peak1_V',        # cleaned
    'ica_peak1_val', 
    'ica_peak2_V', 
    'ica_peak2_val', 
    'ica_area_abs', 
    'degradation_phase',  # NEW: Tracks the voltage phase shift
    'deg_rate_final'      # NEW: The final smoothed, physical degradation rate
]

df_final_export = df_clean[final_columns].copy()

output_filename = 'cleaned_battery_data.csv'
df_final_export.to_csv(output_filename, index=False)

print(f"\nSuccess! File saved as: {output_filename}")
print(f"Final columns included: {list(df_final_export.columns)}")


Success! File saved as: cleaned_battery_data.csv
Final columns included: ['cell', 'cyc', 'capacity_mAh', 'Tmax_C', 'ica_peak1_V', 'ica_peak1_val', 'ica_peak2_V', 'ica_peak2_val', 'ica_area_abs', 'degradation_phase', 'deg_rate_final']
